# Ask Qwen about a transect
Run the code cell, select a sequence or upload up to eight stills, then ask a question. The signed-in OOI session sends the images to the vision model on this Jupyter server. The public [transect review](https://ocean.sig.ai/visual-transect.html) pairs the recorded model outputs with the expert annotations.

In [ ]:
from pathlib import Path
import json, sys
import ipywidgets as w
from IPython.display import display
root = Path.home() / 'data/ocean-platform/visual-transect'
sys.path.insert(0, str(root))
from vision_playground import ask
frames = json.loads((root / 'input/manifest.json').read_text())['frames']
sequence = w.Dropdown(options=[('OceanCV seep transect', 'ooi-ecotype'), ('MBARI Ventana V4289', 'mbari-bs')], description='Sequence')
upload = w.FileUpload(accept='image/jpeg,image/png', multiple=True, description='Upload stills')
question = w.Textarea(value='Describe the seafloor and any recurring animals. Which frames support each observation?', description='Question', layout=w.Layout(width='95%',height='75px'))
go = w.Button(description='Ask Qwen', button_style='primary')
output = w.Output()
def respond(_):
    go.disabled = True
    output.clear_output(wait=True)
    with output:
        try:
            files = upload.value
            if files:
                if isinstance(files, dict):
                    files = [dict(item, name=name) for name, item in files.items()]
                images = [(f['name'], bytes(f['content'])) for f in files]
            else:
                images = [(f"{f['frame_id']} at +{f['timestamp_s']:.2f}s", root / 'input' / f['path']) for f in frames if f['sequence_id'] == sequence.value]
            print(ask(images, question.value))
        except Exception as error:
            print(f'Could not complete inference: {error}')
        finally:
            go.disabled = False
go.on_click(respond)
display(sequence, upload, question, go, output)